# Gaussian Mixture Models: From First Principles

A Gaussian Mixture Model (GMM) represents a dataset as a weighted combination of Gaussian probability distributions. Each distribution is a component with its own mean and covariance; the mixture weight describes how much of the total probability mass is assigned to that component.

GMM is a probabilistic model often used for clustering. It estimates how likely each component is to have generated each observation, so membership can remain uncertain rather than forcing an immediate hard assignment.

## Why Use a Gaussian Mixture Model?

K-Means assigns each observation to its nearest centroid and works best for roughly compact, similarly scaled groups under Euclidean distance. A GMM instead models a probability density. Its components can have different spreads, orientations, and proportions, so it can represent overlapping or elliptical groups.

The trade-off is that GMM makes a distributional assumption: each component is Gaussian. It can represent more varied shapes than spherical K-Means clusters, but it does not automatically find arbitrary curved shapes or density-connected clusters.

## Gaussian Components

A $d$-dimensional Gaussian component is described by a mean vector $\boldsymbol{\mu}_k$ and a covariance matrix $\boldsymbol{\Sigma}_k$. The mean locates the component; the covariance describes its variance in each direction and correlation across features.

A full covariance matrix can represent a tilted ellipsoid. A diagonal covariance permits different feature variances but assumes no within-component feature correlations. A spherical covariance uses one variance value per component and produces spherical contours. Implementations may also offer a tied covariance shared across all components.

A covariance matrix must be positive definite for a nonsingular Gaussian density. Very small clusters or redundant features can create unstable covariance estimates, so fitting may require covariance regularization.

## Soft Assignment and Responsibilities

For observation $\mathbf{x}_i$, the posterior probability that component $k$ generated it is called its responsibility:

$$
r_{ik}=P(z_i=k\mid\mathbf{x}_i)=\frac{\pi_k\,\mathcal{N}(\mathbf{x}_i\mid\boldsymbol{\mu}_k,\boldsymbol{\Sigma}_k)}{\sum_{j=1}^{K}\pi_j\,\mathcal{N}(\mathbf{x}_i\mid\boldsymbol{\mu}_j,\boldsymbol{\Sigma}_j)}
$$

Each responsibility lies between 0 and 1, and the responsibilities for one observation sum to 1. A point may, for example, have responsibility 0.7 for one component and 0.3 for another. A hard cluster label can be made by choosing the component with the largest responsibility, but that discards uncertainty.

These are posterior probabilities under the fitted mixture model. They describe the model's component membership uncertainty; they are not automatically probabilities that an external class label is correct.

## Example: A Soft Membership

Suppose a one-dimensional mixture has two components with equal variance 1, means $\mu_1=0$ and $\mu_2=2$, and weights $\pi_1=0.6$ and $\pi_2=0.4$. For observation $x=1$, the two Gaussian densities are equal because the point is equally far from both means.

The shared density cancels in the responsibility formula, leaving:

$$
r_1=\frac{0.6}{0.6+0.4}=0.6,
\qquad
r_2=\frac{0.4}{0.6+0.4}=0.4
$$

The observation is compatible with both components. The unequal responsibilities come only from the mixture weights in this example.

## Mixture Density and Parameters

For a feature vector $\mathbf{x}\in\mathbb{R}^d$, a Gaussian Mixture Model with $K$ components defines the density:

$$
p(\mathbf{x})=\sum_{k=1}^{K}\pi_k\,\mathcal{N}(\mathbf{x}\mid\boldsymbol{\mu}_k,\boldsymbol{\Sigma}_k)
$$

Each component has a mean vector $\boldsymbol{\mu}_k$ and covariance matrix $\boldsymbol{\Sigma}_k$. Its mixing weight $\pi_k$ is nonnegative, and the weights sum to 1. The covariance controls the component's spread, orientation, and correlations between features.

The density is a weighted sum, not a rule that forces every point into exactly one Gaussian. The model parameters are usually estimated by maximizing the data log-likelihood:

$$
\log p(X)=\sum_{i=1}^{n}\log\left(\sum_{k=1}^{K}\pi_k\,\mathcal{N}(\mathbf{x}_i\mid\boldsymbol{\mu}_k,\boldsymbol{\Sigma}_k)\right)
$$

Because the component identity for each observation is unknown, the likelihood couples latent assignments with the Gaussian parameters. Expectation-Maximization (EM) provides an iterative way to optimize this objective.

## Expectation-Maximization (EM)

The component labels $z_i$ are unknown, so GMM training alternates between estimating memberships and updating the component parameters. This is the Expectation-Maximization algorithm.

**E-step:** Using the current parameters, compute each observation's responsibility $r_{ik}$ with the posterior formula above.

**M-step:** Treat the responsibilities as fractional membership weights. Define the effective component size $N_k=\sum_{i=1}^{n}r_{ik}$. Then update:

$$
\pi_k=\frac{N_k}{n},
\qquad
\boldsymbol{\mu}_k=\frac{1}{N_k}\sum_{i=1}^{n}r_{ik}\mathbf{x}_i,
$$

$$
\boldsymbol{\Sigma}_k=\frac{1}{N_k}\sum_{i=1}^{n}r_{ik}(\mathbf{x}_i-\boldsymbol{\mu}_k)(\mathbf{x}_i-\boldsymbol{\mu}_k)^T
$$

The E-step assigns fractional responsibility; the M-step recomputes mixture weights, means, and covariances. Repeat until the log-likelihood changes very little or another stopping rule is met. Exact EM does not decrease the training likelihood at each iteration, but it can converge to a local optimum rather than the global best solution. Initialization matters, and covariance regularization can help avoid singular estimates.

## Worked EM Iteration in One Dimension

Use observations $x=(0,1,3,4)$ and a two-component, one-dimensional GMM. Initialize both weights to $0.5$, means to $\mu_1=0,\mu_2=4$, and variances to $\sigma_1^2=\sigma_2^2=4$.

### E-step: Compute Responsibilities

Using the current Gaussian densities and equal weights gives:

| $x_i$ | $r_{i1}$ | $r_{i2}$ |
|---:|---:|---:|
| 0 | 0.8808 | 0.1192 |
| 1 | 0.7311 | 0.2689 |
| 3 | 0.2689 | 0.7311 |
| 4 | 0.1192 | 0.8808 |

Each row's responsibilities sum to 1. Summing down the columns gives effective counts $N_1=N_2=2$, so the updated mixture weights remain $\pi_1=\pi_2=2/4=0.5$.

### M-step: Update Means and Variances

The weighted means are:

$$
\mu_1=\frac{0(0.8808)+1(0.7311)+3(0.2689)+4(0.1192)}{2}\approx1.0073,
\qquad
\mu_2\approx2.9927.
$$

For one dimension, covariance is variance. Recomputing the responsibility-weighted squared deviations around these new means gives $\sigma_1^2\approx\sigma_2^2\approx1.5146$. The parameters after one EM iteration are therefore approximately:

| Component | Weight $\pi_k$ | Mean $\mu_k$ | Variance $\sigma_k^2$ |
|---|---:|---:|---:|
| 1 | 0.5 | 1.0073 | 1.5146 |
| 2 | 0.5 | 2.9927 | 1.5146 |

The data log-likelihood increases from approximately $-8.5906$ before the update to $-7.3309$ after it. EM repeats the E- and M-steps with these new parameters until the likelihood or another stopping criterion stabilizes.

## Choosing Components and Initialization

The number of mixture components $K$ must be selected. Fit several plausible values and compare model fit with a complexity penalty, such as the Bayesian Information Criterion (BIC) or Akaike Information Criterion (AIC); lower values are preferred when comparing models fit to the same data under the same likelihood convention. These criteria help balance fit and complexity, but do not guarantee that components correspond to meaningful real-world groups.

EM is sensitive to its starting means, covariances, and weights because it optimizes a non-convex likelihood. A common approach is to initialize with a clustering method such as K-Means and run EM from several starts, retaining a good-likelihood solution. Also check convergence warnings and tiny or nearly singular components.

Feature scaling affects distances and covariance estimation in finite samples, especially with restricted covariance structures or regularization. Choose scaling based on feature units and the intended model assumptions, and fit preprocessing on training data only when evaluating generalization.

## Strengths, Limitations, and When to Use GMM

GMM provides soft membership probabilities, models component-specific covariance, and can represent overlapping elliptical groups. It is useful when uncertainty about cluster membership matters or when the Gaussian-mixture assumption is a reasonable approximation.

Its limitations follow from its assumptions and fitting process:

- The number of components must be chosen or compared across candidate values.
- EM can converge to a local optimum and is sensitive to initialization.
- Gaussian components do not naturally represent arbitrary curved or density-connected shapes.
- Outliers can distort estimated means and covariances; a standard GMM does not provide a dedicated noise label.
- Full covariance models can require many parameters and become unstable with limited data or redundant features.
- Component responsibilities are conditional on the fitted model; they should not automatically be interpreted as calibrated probabilities for an external class.

Inspect convergence, covariance estimates, component sizes, and results across initializations. Compare AIC or BIC for candidate models and, when labels exist for an external task, evaluate that task separately. Use a GMM when its probabilistic Gaussian components match the data and soft assignments are useful; consider DBSCAN for density-connected clusters with noise or K-Means for simpler centroid-based partitions.